Feature Engineering & Data Preprocessing


In [ ]:
import pandas as pd
import numpy as np

# Create a sample DataFrame with missing values
df = pd.DataFrame({
    'feature_A': [1, 2, np.nan, 4, 5],
    'feature_B': ['X', 'Y', 'Z', np.nan, 'Y'],
    'feature_C': [10.5, np.nan, 12.3, 14.0, np.nan]
})

print('Missing values count per column:')
print(df.isnull().sum())
print('\n')
print('Percentage missing per column:')
print(df.isnull().mean() * 100)

# !pip install missingno
# import missingno as msno
# msno.matrix(df)

Missing values count per column:
feature_A    1
feature_B    1
feature_C    2
dtype: int64


Percentage missing per column:
feature_A    20.0
feature_B    20.0
feature_C    40.0
dtype: float64


Step 2 — Fill or drop?

SimpleImputer — filling missing values

In [ ]:
from sklearn.impute import SimpleImputer
import numpy as np
numeric_cols = ['feature_A', 'feature_C']
num_imputer = SimpleImputer(strategy='median')

df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

categorical_cols = ['feature_B']
cat_imputer = SimpleImputer(strategy='most_frequent')

df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])


print('Total missing values after imputation:', df.isnull().sum().sum()) # Should print 0
print('\nDataFrame after imputation:')
print(df)

Total missing values after imputation: 0

DataFrame after imputation:
   feature_A feature_B  feature_C
0        1.0         X       10.5
1        2.0         Y       12.3
2        3.0         Z       12.3
3        4.0         Y       14.0
4        5.0         Y       12.3


Outlier handling

Outliers are extreme values that can distort model training. Two common detection methods:

In [ ]:
# Method 1: IQR (interquartile range) — robust to distribution shape
# Using 'feature_A' as a numerical column for demonstration
Q1 = df['feature_A'].quantile(0.25)
Q3 = df['feature_A'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Keep only rows within bounds
df_clean_iqr = df[(df['feature_A'] >= lower) & (df['feature_A'] <= upper)]
print("DataFrame after IQR outlier removal on 'feature_A':")
print(df_clean_iqr)
print('\n')

# Method 2: Z-score — works well for normally distributed data
from scipy import stats
# Using 'feature_A' for demonst ration
z_scores = stats.zscore(df['feature_A'])
df_clean_zscore = df[abs(z_scores) < 3] # remove rows where |z| > 3
print("DataFrame after Z-score outlier removal on 'feature_A' (threshold 3):")
print(df_clean_zscore)

DataFrame after IQR outlier removal on 'feature_A':
   feature_A feature_B  feature_C
0        1.0         X       10.5
1        2.0         Y       12.3
2        3.0         Z       12.3
3        4.0         W       14.0
4        5.0         W       12.3


DataFrame after Z-score outlier removal on 'feature_A' (threshold 3):
   feature_A feature_B  feature_C
0        1.0         X       10.5
1        2.0         Y       12.3
2        3.0         Z       12.3
3        4.0         W       14.0
4        5.0         W       12.3


Mean vs median

for imputation
Use mean only when your data is roughly symmetric (no big outliers).
Use median when data is skewed (income, house prices, etc.) — the median is not
pulled by extreme values the way the mean is.

sklearn Pipelines

What is data leakage?

Data leakage — the silent model killer

Leakage happens when information from the test set 'leaks' into your training.
Example: if you calculate mean salary on the WHOLE dataset before splitting,
your test rows have already influenced the imputer. The model looks great in
evaluation but fails in production. Pipelines prevent this by fitting only on
training data and applying the same transform to test data.

Building a 3-step pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# Assuming 'df' is the DataFrame from previous steps
# For demonstration, let's define X and y from the existing 'df'
X = df[['feature_A', 'feature_C']] # Using numeric features as X
# Create a dummy target variable 'y' for classification example
# e.g., if feature_A is greater than its median, classify as 1, else 0
y = (df['feature_A'] > df['feature_A'].median()).astype(int)

# Step 1: split data BEFORE building the pipeline
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 2: define the pipeline (a list of (name, transformer) tuples)
pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # fill NaNs
    ('scaler', StandardScaler()), # scale features
    ('model', LogisticRegression(random_state=42)), # train classifier
])

# Step 3: fit ONLY on training data — one call does everything
pipe.fit(X_train, y_train)

# Step 4: evaluate — the pipeline automatically transforms X_test
# using parameters learned from X_train (no leakage!)
print('Accuracy:', pipe.score(X_test, y_test))

# Predict new data in production — just one call
# For demonstration, create a sample X_new based on df's columns
X_new = pd.DataFrame({
    'feature_A': [6.0, np.nan, 8.0],
    'feature_C': [15.0, 16.0, np.nan]
})
predictions = pipe.predict(X_new)
print('Predictions for new data:', predictions)

Accuracy: 1.0
Predictions for new data: [1 1 1]


Lab — Clean a Messy Dataset

Part A — Create the messy dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
np.random.seed(42)
n = 500
# Generate synthetic HR data with intentional problems
df = pd.DataFrame({
'age': np.random.randint(22, 60, n).astype(float),
'salary': np.random.randint(25000, 120000, n).astype(float),
'dept': np.random.choice(['Eng','Sales','HR','Finance'], n),
'education': np.random.choice(['HS','BSc','MSc','PhD'], n),
'years_exp': np.random.randint(0, 35, n).astype(float),
'left_job': np.random.randint(0, 2, n), # target: 1=left, 0=stayed
})
# Inject missing values (~12% each column)
for col in ['age', 'salary', 'dept']:
    df.loc[df.sample(frac=0.12).index, col] = np.nan
# Inject outliers
df.loc[0, 'salary'] = 999999 # extreme outlier
df.loc[1, 'age'] = 150 # impossible age
print('Shape:', df.shape)
print(df.isnull().sum())

Shape: (500, 6)
age          60
salary       60
dept         60
education     0
years_exp     0
left_job      0
dtype: int64


Part B — Baseline (no cleaning)

In [ ]:
# Baseline: just drop rows with missing values — very naive
df_base = df.dropna()
X_base = pd.get_dummies(df_base.drop('left_job', axis=1))
y_base = df_base['left_job']
X_tr, X_te, y_tr, y_te = train_test_split(X_base, y_base,
test_size=0.2, random_state=42)
baseline = LogisticRegression(max_iter=200)
baseline.fit(X_tr, y_tr)
base_acc = accuracy_score(y_te, baseline.predict(X_te))
print(f'Baseline accuracy: {base_acc:.3f}')
# Typically around 0.48-0.52 (near random) due to data loss + no scaling

Baseline accuracy: 0.464


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Part C — Full pipeline (clean model)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
# 1. Define column groups
num_cols = ['age', 'salary', 'years_exp']
cat_cols = ['dept', 'education']
# 2. Fix the impossible outlier BEFORE pipeline
# (domain knowledge: age > 100 is impossible)
df['age'] = df['age'].clip(upper=100)
df['salary'] = df['salary'].clip(upper=200000)
# 3. Split BEFORE any fitting
X = df.drop('left_job', axis=1)
y = df['left_job']
X_tr, X_te, y_tr, y_te = train_test_split(X, y,
test_size=0.2, random_state=42)
# 4. Build sub-pipelines
num_pipe = Pipeline([
('impute', SimpleImputer(strategy='median')),
('scale', StandardScaler()),
])
cat_pipe = Pipeline([
('impute', SimpleImputer(strategy='most_frequent')),
('encode', OneHotEncoder(handle_unknown='ignore',
sparse_output=False)),
])
# 5. Combine
preprocessor = ColumnTransformer([
('num', num_pipe, num_cols),
('cat', cat_pipe, cat_cols),
])
# 6. Full pipeline
model = Pipeline([
('prep', preprocessor),
('clf', LogisticRegression(max_iter=1000)),
])
model.fit(X_tr, y_tr)
clean_acc = accuracy_score(y_te, model.predict(X_te))
print(f'Clean pipeline accuracy: {clean_acc:.3f}')
print(f'Improvement: +{clean_acc - base_acc:.3f}')
# Expect 0.62-0.70 — a clear improvement over baseline

Clean pipeline accuracy: 0.460
Improvement: +-0.004


Part D — Explore results

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = model.predict(X_te)
# Detailed metrics
print(classification_report(y_te, y_pred,
target_names=['Stayed', 'Left']))
# Confusion matrix
print(confusion_matrix(y_te, y_pred))
# Cross-validate to get a more reliable estimate
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f'CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}')

              precision    recall  f1-score   support

      Stayed       0.42      0.33      0.37        48
        Left       0.48      0.58      0.53        52

    accuracy                           0.46       100
   macro avg       0.45      0.46      0.45       100
weighted avg       0.45      0.46      0.45       100

[[16 32]
 [22 30]]
CV accuracy: 0.432 +/- 0.070
